# Graph features

Demonstrates the graph-derived transaction network features (in/out degree, edge frequency, destination historical fraud rate), computed with plain DataFrame aggregations rather than a graph library.

**Prerequisites:** `make sample-data` and `make ingest`.

In [ ]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [ ]:
from pyspark.sql import functions as F

from transaction_risk.features.graph_features import add_graph_features
from transaction_risk.spark.io import read_table

transactions = read_table(spark, '../data/silver/transactions')
with_graph = add_graph_features(transactions)

with_graph.select(
    'nameOrig',
    'nameDest',
    'origin_out_degree',
    'destination_in_degree',
    'edge_frequency',
    'destination_historical_fraud_rate',
).show(10)

In [ ]:
# Destinations that concentrate incoming flows from many distinct accounts are mule-account candidates
(
    with_graph.select('nameDest', 'destination_in_degree', 'destination_historical_fraud_rate')
    .distinct()
    .orderBy(F.col('destination_in_degree').desc())
    .show(10)
)
spark.stop()